In [1]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

from tqdm import tqdm

from urllib.parse import urlsplit, urlunsplit, urlsplit
from urllib.request import urlretrieve

In [2]:
# def save_code(cellml_url, out_path):
#     # Send a GET request to the URL
#     #url += "@@cellml_codegen/Python" #get rid of python code gen
#     name = cellml_url.split("/")[5].split(".")[0]
#     #url = url.replace("slow", "fast") # breaks links
#     #response = requests.get(url)
#     try:
#         filename = Path(urlsplit(cellml_url).path).name
#         save_path = Path(out_path) / filename

#         print(f"Downloading: {cellml_url}")
#         urlretrieve(cellml_url, str(save_path))
#         print(f"Saved to: {save_path}")
#     except Exception as e:
#         print(f"Failed for {cellml_url}\n  {e}")


#     return cellml_url, name

In [3]:
def save_code(cellml_url, out_path, filename=None):
    try:
        original_name = Path(urlsplit(cellml_url).path).name
        filename = filename if filename else original_name

        save_path = Path(out_path) / filename

        print(f"Downloading: {cellml_url}")
        urlretrieve(cellml_url, str(save_path))
        print(f"Saved to: {save_path}")

        return cellml_url, filename[:-7]

    except Exception as e:
        print(f"Failed for {cellml_url}\n  {e}")
        return cellml_url, None

In [4]:
def make_unique(name, existing_names):
    if name not in existing_names:
        return name

    if "." in name:
        base, ext = name.rsplit(".", 1)
        ext = "." + ext
    else:
        base, ext = name, ""

    counter = 2
    new_name = f"{base}_{counter}{ext}"

    while new_name in existing_names:
        counter += 1
        new_name = f"{base}_{counter}{ext}"

    return new_name

In [5]:
# URL of the website you want to crawl
url_list = []
name_list = []
topic_list = []
files_saved = 0

with open("resources/links_fields.txt", "r") as file:
    fields = file.read()
    fields = fields.split("\n")
for topic in fields: #["https://models.physiomeproject.org/synthetic_biology"]: #fields:
    topic_name = topic.split("/")[-1]
    folder_path = f"models_cellml2/{topic_name}"
    # if not Path(folder_path).exists():
    Path(folder_path).mkdir(parents=True, exist_ok=True)
    response = requests.get(topic)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        anchor_tags = soup.find_all("a")
        for tag in anchor_tags:
            link = tag.get("href")
            if link:
                if "/exposure/" in link:
                    link = link[:-5] #gets rid of /view
                    
                    #url, name = save_code(link, folder_path)
                    raw_name = Path(urlsplit(link).path).name
                    unique_name = make_unique(raw_name, name_list)
                    url, name = save_code(link, folder_path, filename=unique_name)
                    url_list.append(url)
                    name_list.append(unique_name)#name)
                    topic_list.append(topic_name)
                    files_saved += 1
    else:
        print("Failed to retrieve the web page.")

clean_name_list = [x[:-7] for x in name_list]

df = pd.DataFrame(list(zip(clean_name_list, url_list, topic_list)))
df.to_csv("cellml_links2.csv")
print(f"Done, {files_saved} saved with {df.shape[0]} rows")

Downloading: https://models.physiomeproject.org/exposure/6c66cc4b25a834b256b9dbb23a3dd9e4/albrecht_colegrove_friel_2002.cellml
Saved to: models_cellml2/calcium_dynamics/albrecht_colegrove_friel_2002.cellml
Downloading: https://models.physiomeproject.org/exposure/ebf69ca24298b28b2361e7d43eb52d6c/albrecht_colegrove_hongpaisan_pivovarova_andrews_friel_2001.cellml
Saved to: models_cellml2/calcium_dynamics/albrecht_colegrove_hongpaisan_pivovarova_andrews_friel_2001.cellml
Downloading: https://models.physiomeproject.org/exposure/210f6601f6461be8443592ff071d2592/baylor_hollingworth_chandler_2002_a.cellml
Saved to: models_cellml2/calcium_dynamics/baylor_hollingworth_chandler_2002_a.cellml
Downloading: https://models.physiomeproject.org/exposure/210f6601f6461be8443592ff071d2592/baylor_hollingworth_chandler_2002_b.cellml
Saved to: models_cellml2/calcium_dynamics/baylor_hollingworth_chandler_2002_b.cellml
Downloading: https://models.physiomeproject.org/exposure/210f6601f6461be8443592ff071d2592/ba